# Práctica 3.3 - Final

Comenzamos creando el sparkContext y otros imports necesarios.
(Una vez en marcha Spark, se puede visualizar la evolución de cada trabajo de Spark en  <http://localhost:4040> )

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .master("local[*]") \
    .appName("Ejemplo pySparkSQL") \
    .config("spark.sql.warehouse.dir", "file:///D:/tmp/spark-warehouse") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
from test_helper import Test
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pyspark.sql.functions import *

## 1. Regresión (Diabetes)

El objetivo de este ejercicio es entrenar un modelo de regresión en Spark para predecir el nivel de diabetes de un conjunto de pacientes. El dataset de diabetes se obtiene de scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html. Este conjunto tiene numerosas variables y la variable a predecir es el nivel de diabetes (target) que en esta mustra toma valores entre 25 y 346. 

Información sobre atributos:

- edad: edad en años
- sexo
- bmi: índice de masa corporal
- bp: presión sanguínea promedio
- s1: tc, colesterol total en suero
- s2: ldl, lipoproteínas de baja densidad
- s3: hdl, lipoproteínas de alta densidad
- s4: tch, colesterol total / HDL
- s5: ltg, logaritmo del nivel de triglicéridos en suero
- s6: glu, nivel de azúcar en sangre


In [ ]:
from sklearn.datasets import load_diabetes

In [ ]:
# Cargar los datos del conjunto del dataset diabetes
diabetes = # <RELLENAR>

# Nos quedamos en X con data y en y target
X = # <RELLENAR>
y = # <RELLENAR>

# Convertir las matrices NumPy en un dataframe de pandas
df_pandas = pd.DataFrame(np.hstack([X, y.reshape(-1, 1)]), columns=list(diabetes.feature_names) + ['target'])

# Convertir el dataframe de pandas en un dataframe de Spark
df = # <RELLENAR>

# Número mínimo y máximo en target.
min_target, max_target = np.min(diabetes.target), np.max(diabetes.target)

In [ ]:
Test.assertEquals(min_target, 25.0, "Número mínimo incorrecto")
Test.assertEquals(max_target, 346.0, "Número máximo incorrecto")

Una vez cargados los daros imprimimos el esquema del dataframe y visualizamos los primeros datos:

In [ ]:
# Mostramos el esquema del dataset.
# <RELLENAR>

In [ ]:
# Enseñamos los 5 primeros valores.
# <RELLENAR>

### Previsualización de datos
Podemos observar cierta tendencia entre el nivel de diabetes y el índice de masa corporal. 

In [ ]:
# Cargamos todos los datos del dataframe.
tuples = list(map(lambda row: (row["bmi"], row["target"]), # <RELLENAR>))

# Con el siguiente plot pintamos los datos de target en relación a la variable pes
plt.scatter(list(zip(*tuples))[0], list(zip(*tuples))[1])
plt.ylabel('target')
plt.xlabel('Indice de masa corporal')
plt.show()

Graficando otras variables no se observa tan clara la tendencia. Vemos por ejemplo para la edad que no se observa una tendencia clara.

In [ ]:
# Cargamos todos los datos del dataframe.
tuples = list(map(lambda row: (row["age"], row["target"]), # <RELLENAR>))

# Con el siguiente plot pintamos los datos de target en relación a la variable pes
plt.scatter(list(zip(*tuples))[0], list(zip(*tuples))[1])
plt.ylabel('target')
plt.xlabel('Edad')
plt.show()

No obstante para una primera aproximación vamos a utilizar todas las variables disponibles.

Dividamos el dataframe en train y test, 70% y 30% respectivamente.

In [ ]:
# Separamos con semilla 1234.
seed = 1234

# Dividir el dataset en el 70% para train y el 30% para test
(train, test) = # <RELLENAR>

# Cómo vamos a usar bastante este dataset lo cacheamos. ¡¡¡CUIDADO LABORATORIO VIRTUAL!!!
# <RELLENAR>

print("Tenemos %d ejemplos en training y %d en test." % (train.count(), test.count()))

In [ ]:
Test.assertEquals(train.count(), 296, "Número en train incorrecto")
Test.assertEquals(test.count(), 146, "Número en test incorrecto")
Test.assertEquals(train.is_cached, True, 'Training no cacheado')

In [ ]:
from pyspark.ml.feature import VectorAssembler, VectorIndexer

# Creamos el VectorAssembler y añadimos todas las variables excepto target.
input_col = # <RELLENAR>
input_col.# <RELLENAR>
assembler = VectorAssembler(inputCols=input_col, outputCol="rawFeatures")

# Identificamos las características categóricas y las indexamos
vectorIndexer = VectorIndexer(inputCol="rawFeatures", outputCol="features", maxCategories=4)

### Modelo de regresión GBT 

Creamos el modelo de regresión `GBTRegressor` Gradient Boosted Trees, que combina múltiples árboles de decisión  para crear un modelo más robusto y preciso. En particular, GBTRegressor se basa en un enfoque de boosting, donde los árboles se construyen secuencialmente, y cada árbol intenta corregir los errores de los árboles anteriores en el conjunto. GBT toma un vector de características y las etiquetas como entradas y aprende un modelo de regresión que luego podremos utilizar para predecir nuevos ejemplos. Como ya hemos comentado vamos a utilizar todas las variables disponibles para entrenar el modelo de regresión.

In [ ]:
from pyspark.ml.regression import GBTRegressor

# Toma las columna "features" y aprende a predecir "target"
gbt = # <RELLENAR>

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# Definición del grid de parámetros para la búsqueda:
paramGrid = ParamGridBuilder()\
  .addGrid(gbt.maxDepth, [2, 5])\
  .addGrid(gbt.maxIter, [10, 20, 100, 150])\
  .build()

# Definimos una métrica para la evaluación, usando cómo métrica rmse
evaluator = RegressionEvaluator(# <RELLENAR>)

# Declaramos el CrossValidator, que requiere de un estimator, un evaluator y los estimatorParamMaps (parámetros)
# Consulta http://spark.apache.org/docs/2.4.5/api/python/pyspark.ml.html#pyspark.ml.tuning.CrossValidator
cv = CrossValidator(estimator=gbt,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=2)

Finalmente podemos unir nuestra fase de procesamiento de características con el entrenamiento del modelo en una única Pipeline con las tres fases anteriores.

In [ ]:
from pyspark.ml import Pipeline

pipeline = # <RELLENAR>

### Entrenamiento de la Pipeline

Ahora que ya tenemos el flujo de datos definido, podemos entrenar la Pipeline completa con una única llamada a fit(). Esta llamada llevará a cabo el preprocesamiento de características, el ajuste de los parámetros y el entrenamiento del modelo final.

Como resultado obtendremos una Pipeline ajustada (PipelineModel).

In [ ]:
# Entrenar la pipeline con fit y el conjunto de train
pipelineModel = # <RELLENAR>

Obtenemos las predicciones en train (pred_train) y en test(pred_test):

In [ ]:
pred_train = # <RELLENAR>
pred_test = # <RELLENAR>

Podemos ver las predicciones de unos pocos registros para intuir que tal funciona nuestro modelo.

In [ ]:
pred_train.select('bmi', 'target', 'prediction').show(10)

### Evaluacion

Evaluamos los resultados. Utilizaremos RMSE. 

In [ ]:
# Utiliza el evaluator con su método evaluate para obtener el RMSE sobre las predicciones de train y test
rmse_train = # <RELLENAR>
print("RMSE en train: %g" % rmse_train)

rmse_test = # <RELLENAR>
print("RMSE en test: %g" % rmse_test)

In [ ]:
Test.assertEquals(rmse_train, 52.3492, "RMSE en train incorrecto")
Test.assertEquals(rmse_test, 59.8296, "RMSE en test incorrecto")

### Modelo GBT simplificado

Vamos a simplificar el modelo eliminando algunas de las variables ya que podemos deducir que ciertas variables no aportan mucho al modelo. Por ejemplo no utilizaremos el sexo y algunos desgloses del nivel de colesterol.

In [ ]:
# Creamos el VectorAssembler, nos quedamos con las variables: 'bmi', 'bp', 's4', 's5' y 's6'
input_col = ['bmi', 'bp', 's4', 's5', 's6']
assembler = VectorAssembler(inputCols=input_col, outputCol="rawFeatures")

# Identificamos las características categóricas y las indexamos 
vectorIndexer = VectorIndexer(inputCol="rawFeatures", outputCol="features", maxCategories=4)

In [ ]:
from pyspark.ml.regression import GBTRegressor

# Usando GBTRegressor predecimos sobre "target"
gbt = # <RELLENAR>

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# Definición del grid de parámetros para la búsqueda:
paramGrid = ParamGridBuilder()\
  .addGrid(gbt.maxDepth, [2, 5])\
  .addGrid(gbt.maxIter, [10, 20])\
  .build()

# Definimos una métrica para la evaluación
evaluator = RegressionEvaluator(metricName="rmse", labelCol=gbt.getLabelCol(), predictionCol=gbt.getPredictionCol())

# Declaramos el CrossValidator, que requiere de un estimator, un evaluator y los estimatorParamMaps (parámetros)
# Consulta http://spark.apache.org/docs/2.4.5/api/python/pyspark.ml.html#pyspark.ml.tuning.CrossValidator
cv = CrossValidator(estimator=gbt,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=2)

In [ ]:
# Creamos una Pipeline con los tres pasos anteriores.
pipeline = # <RELLENAR>

In [ ]:
# Entrenar la pipeline sobre el conjunto de train
pipelineModel = # <RELLENAR>

In [ ]:
# Nuevas predicciones sobre train y test.
pred_train2 = # <RELLENAR>
pred_test2 = # <RELLENAR>

In [ ]:
# Evaluamos RMSE del segundo modelo sobre las predicciones de la celda anterior.
rmse_train2 = # <RELLENAR>
print("RMSE en train: %g" % rmse_train2)

rmse_test2 = # <RELLENAR>
print("RMSE en test: %g" % rmse_test2)

In [ ]:
Test.assertEquals(rmse_train2, 52.9783, "RMSE en train incorrecto")
Test.assertEquals(rmse_test2, 60.8939, "RMSE en test incorrecto")

In [ ]:
# Graficar los valores reales vs las predicciones en test
# En un buen ajuste los puntos graficados estarían muy próximos a la diagonal
x = # <RELLENAR> # Todos los datos del campo target de las nuevas predicciones de train.
y = # <RELLENAR> # Todos los datos del campo prediction de las nuevas predicciones de train.
x2 = # <RELLENAR> # Todos los datos del campo target de las nuevas predicciones de test.
y2 = # <RELLENAR> # Todos los datos del campo target de las nuevas predicciones de test.

plt.figure(figsize=(10, 6))
plt.scatter(x,y , alpha=0.5, label='Modelo 1')
plt.scatter(x2,y2 , alpha=0.5, c='green',label='Modelo 2')
plt.legend()
plt.xlabel('Valores Reales')
plt.ylabel('Predicciones')
plt.plot(x,x,color='red', linestyle='-')  # Línea diagonal
plt.title('Comparación entre Valores Reales y Predicciones')
plt.grid(True)
plt.show()

No se observan excesivas diferencias entre los 2 modelos. En el modelo mas simple se obtiene un rmse de 60.9 algo peor que en el modelo con todas las variables que era de 59.8.

Finalmente nos quedaremos con el modelo de regresión lineal ya que obtenemos un RMSE menor y un mejor ajuste tal y como se observa en el gráfico anterior.

## 2. Clasificación ( Tipos de bosque - dataset covtype )


El objetivo de este ejercicio es entrenar un modelo de clasificación en Spark para predecir el tipo de vegetación. El dataset covtype se obtiene de scikit-learn. Este conjunto es muy extenso ya que tiene 581.012 filas y también tiene numerosas variables (54). La clase a predecir tiene 7 valores posibles:

1. Spruce/Fir
2. Lodgepole Pine
3. Ponderosa Pine
4. Cottonwood/Willow
5. Aspen
6. Douglas-fir
7. Krummholz

In [ ]:
from sklearn.datasets import fetch_covtype

# Cargar los datos del conjunto de datos fetch_covtype
covtype = # <RELLENAR>

# Los separamos en X (data) e y (target).
X = # <RELLENAR>
y = # <RELLENAR>

In [ ]:
Test.assertEquals(X.shape, (581012, 54), "Carga de datos incorrecta")

Podemos observar que tenemos 581.012 registros/filas y 54 variables/columnas. Como son muchas variables no vamos a realizar un analisis detallado del significado de éstas. 

Las 4 primeras variables por ejemplo son las siguientes:

* Elevation (Elevación): Es la elevación en metros sobre el nivel del mar. Esta característica indica la altitud del terreno en la ubicación dada.

* Aspect (Orientación): Se refiere a la dirección hacia la cual está orientada una pendiente. Se mide en grados, representando el ángulo con respecto al norte en sentido horario.

* Slope (Pendiente): Es la inclinación del terreno en grados. Indica la tasa de cambio en la elevación con respecto a la distancia horizontal.

* Horizontal_Distance_To_Hydrology (Distancia horizontal a hidrología): Es la distancia horizontal desde el punto en cuestión hasta la característica de hidrología más cercana, como un río o un arroyo, medida en metros.



Los tipos de bosque son 7, por lo tanto númeramos las clases del 1 a 7.

In [ ]:
class_labels = covtype.target
unique_class_labels = np.unique(class_labels)
print("Clases:", unique_class_labels)

In [ ]:
Test.assertEquals(unique_class_labels, [1, 2, 3, 4, 5, 6, 7], "Carga de datos incorrecta")

Como tenemos muchos registros y el laboratorio virtual no responde de manera ágil vamos a seleccionar de forma estratificada el 1 % de los registros y crear un dataframe en spark con ellos. 

In [ ]:
from sklearn.model_selection import train_test_split

# Para las pruebas nos quedamos con el 1% de los datos, estratificados por y (variable objetivo)
Xred, _, yred, _ = # <RELLENAR>

# Convertir las matrices NumPy en un dataframe de pandas
# En lugar de usar como nombre de etiqueta 'Cover_Type' usaremos 'class'
df = pd.DataFrame(np.hstack([Xred, yred.reshape(-1, 1)]), columns=covtype.feature_names + ['class'])

# Convertir el dataframe de pandas en un dataframe de Spark
df = # <RELLENAR>

Podemos comprobar que el dataframe reducido con el 1% de los datos tiene la misma distribución en cuanto a porcentaje de cada clase que el original

In [ ]:
# Obtener los nombres únicos de las etiquetas de clase y sus frecuencias
unique_classes, class_counts = np.unique(y, return_counts=True)
unique_classes_red, class_counts_red = np.unique(yred, return_counts=True)

# Imprimir los nombres de las clases y la cantidad de registros para cada clase
for class_label, class_count, class_count_red in zip(unique_classes, class_counts, class_counts_red):
    print(f"Clase {class_label}: {np.round(class_count/y.shape[0]*100)} % registros conjunto total / {np.round(class_count_red/yred.shape[0]*100)} % registros conjunto reducido")


Visualizamos uno de los registros del Dataframe

In [ ]:
# <RELLENAR>

Dividimos el dataframe en train y test, 70% y 30% respectivamente. Cacheamos en conjunto de train para tener mas accesibles los datos que vamos a reutilizar en los distintos entrenamientos de modelos.

In [ ]:
# Separamos con semilla 1234 y cacheamos train.
seed = 1234
(train, test) = # <RELLENAR>
# <RELLENAR>

In [ ]:
input_col = df.columns 
input_col.remove('class') 
assembler = VectorAssembler(inputCols=input_col, outputCol="rawFeatures")

# Identificamos las características categóricas y las indexamos 
vectorIndexer = VectorIndexer(inputCol="rawFeatures", outputCol="features", maxCategories=4)

In [ ]:
from pyspark.ml.classification import LogisticRegression

# Regresión logística con 20 iteraciones y parámetro de regularización = 0.01
lr = LogisticRegression(maxIter=20, regParam=0.01, featuresCol='features', labelCol='class', predictionCol='prediction')

# Creamos la pipeline como una lista de fases
pipeline = # <RELLENAR>

Entrenamiento del modelo

In [ ]:
spark.sparkContext.setLogLevel("ERROR")
model = # <RELLENAR>

Obtenemos las predicciones en train y test

In [ ]:
predictionTrain = # <RELLENAR>
predictionTest = # <RELLENAR>

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
multi_evaluator = MulticlassClassificationEvaluator(predictionCol='prediction',labelCol='class')

# Utiliza el evaluador creado
# Calcular precision, recall y F-score en train y test
train_precision = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "weightedPrecision"})
train_recall = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "weightedRecall"})
train_f1score = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "f1"})                                        
test_precision = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "weightedPrecision"})
test_recall = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "weightedRecall"})
test_f1score = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "f1"})


In [ ]:
# Imprimir las métricas
print('Train Precision:', train_precision)
print('Train Recall:', train_recall)
print('Train F1-score:', train_f1score)

print("\n")

print('Test Precision:', test_precision)
print('Test Recall:', test_recall)
print('Test F1-score:', test_f1score)

Vamos a intentar mejorar nuestro modelo con ajuste de hiperparametros mediante validación cruzada 

In [ ]:
lr = LogisticRegression(featuresCol='features', labelCol='class', predictionCol='prediction')
input_col = df.columns 
input_col.remove('class') 
assembler = VectorAssembler(inputCols=input_col, outputCol="features")

# En este caso utilizaremos un pipeline sin vectorIndexer
pipeline = # <RELLENAR>

# evaluador de multiclase con métrica weightedRecall
evaluator = MulticlassClassificationEvaluator(predictionCol='prediction',labelCol='class',metricName='f1')

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder,CrossValidator
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.01, 0.1, 0.2])
             .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
             .addGrid(lr.maxIter, [10, 50, 100,500])
             .addGrid(lr.threshold, [0.4, 0.5, 0.6])
             .addGrid(lr.fitIntercept, [True, False])
             .build())

In [ ]:
# Llamamos el validaro cruzado.
cv = CrossValidator(# <RELLENAR>)

# Entrenamos el modelo.
cvModel = # <RELLENAR>

Obtenemos las predicciones en train y test

In [ ]:
predictionTrain = # <RELLENAR>
predictionTest = # <RELLENAR>

In [ ]:
train_precision = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "weightedPrecision"})
train_recall = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "weightedRecall"})
train_f1score = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "f1"})                                        
test_precision = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "weightedPrecision"})
test_recall = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "weightedRecall"})
test_f1score = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "f1"})

In [ ]:
# Imprimir las métricas
print('Train Precision:', train_precision)
print('Train Recall:', train_recall)
print('Train F1-score:', train_f1score)
print("\n")
print('Test Precision:', test_precision)
print('Test Recall:', test_recall)
print('Test F1-score:', test_f1score)

In [ ]:
# Extraemos los parametros del modelo optimizado mediante cv
paramMap = cvModel.bestModel.stages[1].extractParamMap()

# Imprimimos los resultados.
print("Parametros del modelo:")
for param, value in paramMap.items():
    #RELLENAR
    print('Parametro',param,':',value)

Con un modelo optimizado mediante validación cruzada conseguimos una pequeña mejora pasando en test a un F1-score de cerca de 0,7. 

Los hiperparametros elegidos mediante validación cruzada son:
* regParam: 0.01
* elasticNetParam: 0.0
* maxIter: 500
* threshold: 0.6
* fitIntercept: True

NOTA:

Una vez testeado nuestro modelo podriamos aprovechar el potencial de spark realizando el entrenamiento con la totalidad de los datos (581.012 registros) ya que para el ejercicio solo hemos utilizado el 1% de los mismos debido a las limitaciones del laboratorio virtual.  

### Clasificación  PCA + Regresión logística

Vamos a entrenar otro modelo. Creamos una pipeline que utiliza el assembler, PCA con 20 componentes principales y regresión logística. 

In [ ]:
from pyspark.ml.feature import PCA

# assembler
assembler = VectorAssembler(inputCols=input_col, outputCol="features")

# Número de componentes para PCA 
num_components = 20
pca = PCA(k=num_components, inputCol="features", outputCol="pca_features")

# Regresión logística con 20 iteraciones y parámetro de regularización = 0.01
lr = LogisticRegression(maxIter=500, regParam=0.01, featuresCol='pca_features', labelCol='class', predictionCol='prediction')

# Crear el pipeline
pipeline = Pipeline(stages=[assembler, pca, lr])

# Entrenamiento
pipeline_model = pipeline.fit(train)

# Predicciones
predictionTrain = pipeline_model.transform(train)
predictionTest = pipeline_model.transform(test)

In [ ]:
train_precision = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "weightedPrecision"})
train_recall = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "weightedRecall"})
train_f1score = multi_evaluator.evaluate(predictionTrain, {multi_evaluator.metricName: "f1"})                                        
test_precision = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "weightedPrecision"})
test_recall = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "weightedRecall"})
test_f1score = multi_evaluator.evaluate(predictionTest, {multi_evaluator.metricName: "f1"})

In [ ]:
# Imprimir las métricas
print('Train Precision:', train_precision)
print('Train Recall:', train_recall)
print('Train F1-score:', train_f1score)
print("\n")
print('Test Precision:', test_precision)
print('Test Recall:', test_recall)
print('Test F1-score:', test_f1score)

Con PCA se ha logrado simplificar el modelo, lo que puede mejorar la eficiencia computacional y reducir el riesgo de sobreajuste. Sin embargo, perdemos capacidad predictiva del modelo.